In [1]:
from typing import List, Dict
import random

# Each device (bin) has a capacity and current usage
devices = [
    {"name": "cpu", "type": "cpu", "mem_limit": 8000, "used": 2000, "latency": 200},   # ms
    {"name": "gpu0", "type": "gpu", "mem_limit": 16000, "used": 8000, "latency": 50},
    {"name": "gpu1", "type": "gpu", "mem_limit": 16000, "used": 4000, "latency": 40},
    {"name": "multi_gpu", "type": "multi_gpu", "mem_limit": 32000, "used": 10000, "latency": 25},
]

# Inference requests with model size and expected benefit (value = saved latency)
inference_tasks = [
    {"id": "task1", "model": "ResNet18", "size": 1000, "latency_saved": 150},
    {"id": "task2", "model": "ResNet152", "size": 5000, "latency_saved": 300},
    {"id": "task3", "model": "BERT", "size": 12000, "latency_saved": 450},
    {"id": "task4", "model": "InceptionV3", "size": 7000, "latency_saved": 250},
]

def rule_based_filter(task, device):
    # RULE 1: skip if device cannot fit model
    if device["mem_limit"] - device["used"] < task["size"]:
        return False
    # RULE 2: if task is big, prefer GPU or Multi-GPU
    if task["size"] > 8000 and device["type"] == "cpu":
        return False
    return True

def latency_aware_bin_packing(tasks: List[Dict], devices: List[Dict]):
    assignments = []
    for task in tasks:
        best_device = None
        best_gain = 0

        for device in devices:
            if not rule_based_filter(task, device):
                continue
            # Simulate expected latency gain = baseline_latency - device_latency
            gain = task["latency_saved"] - device["latency"]
            if gain > best_gain:
                best_gain = gain
                best_device = device

        if best_device:
            assignments.append((task["id"], best_device["name"], best_gain))
            best_device["used"] += task["size"]  # update usage

    return assignments

# Run
assignments = latency_aware_bin_packing(inference_tasks, devices)
for a in assignments:
    print(f"Assign {a[0]} to {a[1]} with expected latency gain: {a[2]} ms")


Assign task1 to multi_gpu with expected latency gain: 125 ms
Assign task2 to multi_gpu with expected latency gain: 275 ms
Assign task3 to multi_gpu with expected latency gain: 425 ms
Assign task4 to gpu1 with expected latency gain: 210 ms
